1. vs코드로 돌리는거 추천
2. 캐글로 우회해서 메타 데이터 수집했는데 다른 방법 있으면 공유 부탁
3. 아마 결측치 나오는건 노션에 올린 링크로 따로 채우시면 될 듯 합니다! 
4. 스포티파이 / 지니어스 / 캐글 api키 입력되어있습니다.

# Spotify Api

In [1]:
!pip install spotipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [spotipy]


In [2]:
artists = [
    ("Swings", "스윙스"),
    ("Simon Dominic", "사이먼 도미닉"),
    ("Loco", "로꼬"),
    ("Jay Park", "박재범"),
    ("pH-1", "피에이치원"),
    ("Paloalto", "팔로알토"),
    ("The Quiett", "더 콰이엇"),
    ("Dok2", "도끼"),
    ("Coogie", "쿠기"),
    ("ASH ISLAND", "애쉬 아일랜드"),
    ("Sik-K", "식케이"),
    ("SUPERBEE", "수퍼비"),
    ("UNEDUCATED KID", "언에듀케이티드 키드"),
]


In [3]:
import requests
import pandas as pd
import time
import base64

# 인증 정보
CLIENT_ID = '1167543a81304b0aaf96e4b62ad9a500'.strip()
CLIENT_SECRET = 'b82db2263ff44fad8da54c1a9348796b'.strip()

def get_access_token(cid, csecret):
    auth_url = 'https://accounts.spotify.com/api/token'
    auth_header = base64.b64encode(f"{cid}:{csecret}".encode('utf-8')).decode('utf-8')
    headers = {'Authorization': f'Basic {auth_header}'}
    data = {'grant_type': 'client_credentials'}
    res = requests.post(auth_url, headers=headers, data=data)
    return res.json().get('access_token')

def collect_krap_final(artist, tracks_per_artist=30):
    token = get_access_token(CLIENT_ID, CLIENT_SECRET)
    if not token:
        print(" 토큰 발급 실패!")
        return pd.DataFrame()

    headers = {'Authorization': f'Bearer {token}'}
    all_tracks = []

    print(f"--- [{artist}] 수집 시작 (목표: {tracks_per_artist}곡) ---")

    step = 10
    offset = 0
    max_offset = 100  # 무한루프 방지

    while len(all_tracks) < tracks_per_artist and offset < max_offset:
        search_url = "https://api.spotify.com/v1/search"
        params = {
            'q': f'artist:{artist}',
            'type': 'track',
            'limit': step,
            'offset': offset
        }

        try:
            res = requests.get(search_url, headers=headers, params=params)

            if res.status_code != 200:
                print(f"오류 발생: {res.status_code}, {res.text}")
                break

            data = res.json()
            items = data['tracks']['items']

            if not items:
                print("더 이상 검색 결과가 없습니다.")
                break

            for t in items:
                # 피처링 제외: 첫 번째 아티스트가 본인인 경우만
                if artist.lower() in t['artists'][0]['name'].lower():
                    all_tracks.append({
                        'title': t['name'],
                        'artist': t['artists'][0]['name'],
                        'spotify_id': t['id']
                    })
                    if len(all_tracks) >= tracks_per_artist:
                        break

            print(f"진행 중: {len(all_tracks)}곡 수집 완료... (offset: {offset})")
            offset += step
            time.sleep(3)

        except Exception as e:
            print(f"에러: {e}")
            break

    return pd.DataFrame(all_tracks)

In [4]:
all_dfs = []
for eng_name, kor_name in artists:
    for query_name in [eng_name, kor_name]:
        print(f"\n수집 중: {query_name}")
        df_artist = collect_krap_final(query_name, tracks_per_artist=30)
        if not df_artist.empty:
            all_dfs.append(df_artist)
            print(f"{query_name} — {len(df_artist)}곡 수집")
            break
        print(f"{query_name} — 결과 없음, 한글명으로 재시도...")
    time.sleep(3)

if all_dfs:
    df_krap = pd.concat(all_dfs, ignore_index=True).drop_duplicates(subset=['spotify_id'])
    print(f"\n전체 수집 완료: {len(df_krap)}곡")
    df_krap.to_csv("korean_rap_titles.csv", index=False, encoding='utf-8-sig')
else:
    print("수집된 데이터가 없습니다.")


수집 중: Swings
--- [Swings] 수집 시작 (목표: 30곡) ---
진행 중: 4곡 수집 완료... (offset: 0)
진행 중: 6곡 수집 완료... (offset: 10)
진행 중: 11곡 수집 완료... (offset: 20)
진행 중: 15곡 수집 완료... (offset: 30)
진행 중: 18곡 수집 완료... (offset: 40)
진행 중: 24곡 수집 완료... (offset: 50)
진행 중: 28곡 수집 완료... (offset: 60)
진행 중: 30곡 수집 완료... (offset: 70)
Swings — 30곡 수집

수집 중: Simon Dominic
--- [Simon Dominic] 수집 시작 (목표: 30곡) ---
진행 중: 2곡 수집 완료... (offset: 0)
진행 중: 8곡 수집 완료... (offset: 10)
진행 중: 12곡 수집 완료... (offset: 20)
진행 중: 13곡 수집 완료... (offset: 30)
진행 중: 16곡 수집 완료... (offset: 40)
진행 중: 18곡 수집 완료... (offset: 50)
진행 중: 23곡 수집 완료... (offset: 60)
진행 중: 25곡 수집 완료... (offset: 70)
진행 중: 29곡 수집 완료... (offset: 80)
진행 중: 30곡 수집 완료... (offset: 90)
Simon Dominic — 30곡 수집

수집 중: Loco
--- [Loco] 수집 시작 (목표: 30곡) ---
진행 중: 3곡 수집 완료... (offset: 0)
진행 중: 5곡 수집 완료... (offset: 10)
진행 중: 6곡 수집 완료... (offset: 20)
진행 중: 6곡 수집 완료... (offset: 30)
진행 중: 8곡 수집 완료... (offset: 40)
진행 중: 10곡 수집 완료... (offset: 50)
진행 중: 12곡 수집 완료... (offset: 60)
진행 중: 16곡 수집 완료... (of

In [5]:
print(df_krap.groupby('artist')['title'].count().sort_values()) # 수집된 곡 수 확인

artist
죠지, 쿠기 (Coogie)     1
Paloalto           15
Loco               22
pH-1               26
Coogie             29
ASH ISLAND         30
DOK2               30
Jay Park           30
SUPERBEE           30
Sik-K              30
Simon Dominic      30
Swings             30
The Quiett         30
UNEDUCATED KID     30
Name: title, dtype: int64


# 지니어스 데이터셋


In [6]:
!pip install lyricsgenius

In [8]:
import lyricsgenius
import pandas as pd
import time
import re
import csv  

# 2. Genius 인증
GENIUS_TOKEN = 'K9PVrXo6yuJSakz2TveXXtZZNxmYwiL7CMTJQy8Itcm9D0kWkd5eVPYNpaevCZqL'
genius = lyricsgenius.Genius(GENIUS_TOKEN, sleep_time=2.0, retries=3, timeout=20)
genius.headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

def clean_lyrics_text(text):
    if not text: return ""
    text = re.sub(r'\d*Embed$', '', text)
    lines = text.split('\n')
    if len(lines) > 1 and 'Lyrics' in lines[0]:
        return '\n'.join(lines[1:])
    return text

# 제목 정제 함수 (매칭률 향상을 위해 추가)
def clean_title_for_search(title):
    return re.sub(r'\(.*?\)|\[.*?\]', '', title).strip()

lyrics_results = []

print(f"--- 총 {len(df_krap)}곡 가사 수집 시작 ---")

for index, row in df_krap.iterrows():
    try:
        # 검색용 제목/가수 정제
        search_title = clean_title_for_search(row['title'])
        clean_artist = re.sub(r'\(.*\)', '', row['artist']).strip()

        print(f" [{index+1}/{len(df_krap)}] 검색 중: {search_title} - {clean_artist}")
        song = genius.search_song(search_title, clean_artist)

        if song:
            lyrics_results.append(clean_lyrics_text(song.lyrics))
            print(f"✅ 성공")
        else:
            lyrics_results.append("")
            print(f"❌ 가사 없음")

    except Exception as e:
        lyrics_results.append("")
        print(f"⚠️ 에러 발생: {e}")
        time.sleep(5) 

    time.sleep(1.5)

#결과 반영
df_krap['lyrics'] = lyrics_results
df_final = df_krap[df_krap['lyrics'] != ""].copy()

df_final.to_csv(
    "korean_rap_lyrics.csv", 
    index=False, 
    encoding='utf-8-sig', 
    quoting=csv.QUOTE_ALL 
)

print(f"\n 수집 완료! 총 {len(df_final)}곡 저장되었습니다.")

--- 총 363곡 가사 수집 시작 ---
 [1/363] 검색 중: 듣고 있어? - Swings
✅ 성공
 [2/363] 검색 중: Not Enough - Swings
✅ 성공
 [3/363] 검색 중: Invasion - Swings
✅ 성공
 [4/363] 검색 중: No One Likes Us - Swings
✅ 성공
 [5/363] 검색 중: PORN FLICK - Swings
✅ 성공
 [6/363] 검색 중: Villain - Swings
✅ 성공
 [7/363] 검색 중: Fake Rock Star - Swings
✅ 성공
 [8/363] 검색 중: Watch - Swings
✅ 성공
 [9/363] 검색 중: A Real Lady - Swings
✅ 성공
 [10/363] 검색 중: WAKEUP - Swings
✅ 성공
 [11/363] 검색 중: Want It - Swings
✅ 성공
 [12/363] 검색 중: 양아치 - Swings
✅ 성공
 [13/363] 검색 중: step back b**** 'cause I'm too fye - Swings
✅ 성공
 [14/363] 검색 중: Upgrade 2020 - Swings
✅ 성공
 [15/363] 검색 중: Warrior - Swings
✅ 성공
 [16/363] 검색 중: 2007 - Swings
✅ 성공
 [17/363] 검색 중: Fosho-! - Swings
✅ 성공
 [18/363] 검색 중: Whistle - Swings
✅ 성공
 [19/363] 검색 중: Shit Is Real - Swings
✅ 성공
 [20/363] 검색 중: Bulldozer - Swings
✅ 성공
 [21/363] 검색 중: Pot - Swings
✅ 성공
 [22/363] 검색 중: Keep Going - Swings
✅ 성공
 [23/363] 검색 중: Perky - Swings
✅ 성공
 [24/363] 검색 중: SIDE - Swings
✅ 성공
 [25/363] 검색 중: A Real Ma

In [9]:
# 1. 메모리에 데이터가 있는지 최종 확인
if 'df_final' in locals() and not df_final.empty:
    # 2. 첫 번째 곡 정보 가져오기
    first_song = df_final.iloc[0]
    
    print(f"데이터셋 확인 (총 {len(df_final)}곡 수집됨)")
    print("-" * 50)
    print(f"제목: {first_song['title']}")
    print(f"아티스트: {first_song['artist']}")
    print("-" * 50)
    
    # 3. 가사 길이 확인
    lyrics_sample = first_song['lyrics']
    print(f"가사 길이: {len(lyrics_sample)}자")
    print("-" * 50)
    print(lyrics_sample[:500]) # 앞부분 500자만 출력
    print("\n... (이하 생략) ...")
    print("-" * 50)
else:
    print("수집된 데이터가 없습니다. 지니어스 수집 셀을 다시 실행해 보세요!")

데이터셋 확인 (총 347곡 수집됨)
--------------------------------------------------
제목: 듣고 있어?
아티스트: Swings
--------------------------------------------------
가사 길이: 1223자
--------------------------------------------------
[Verse 1]
벌써 2년이 지났지
세상은 많이 바뀐 듯해
너와의 인연 끝에서
네게 미련 없을 거라고 확신했었던 나지
날 사랑한다고 말했어, 그때 니 말을 믿었을 걸
며칠 전 그 누나 만났다며
너 잘 지내냐고 물어보려다 말았어
꼭 헤어지고 나면 남자들은 바뀌어
계속 연락했더니 넌 번호를 바꿨어
Good girl, 난 니가 자랑스러워
니 남자친구가 진심으로 부러워
만나면 걜 때릴지도 몰라
내가 이러니 넌 비밀리에 사귄 거야
걔가 너를 울렸으면 좋겠어
나보다 100배, 아니, 1000배 더 못 되서
근데 그건 불가능해, 그건 악마지
연락 할 수 있는 방법은 하나지
나 연기 실력 늘었어, 속으론 울고 있어
이 노랜 떠야 돼, 널 위해, 듣고 있어?

[Chorus]
I just wanted to know if you still hate me
한 번이라도 나를 그리워 했니?
이 노랠 듣고 있다면 고민하지마
그땐 몰랐지만 난 너의 중독자야
For your love, your love, yo

... (이하 생략) ...
--------------------------------------------------


# Kaggle 이용

In [10]:
# 1. kagglehub 설치
!pip install kagglehub -q

import kagglehub
import os

# 2. API 키 설정
os.environ['KAGGLE_USERNAME'] = 'ryongryong'
os.environ['KAGGLE_KEY'] = '3c005f773e42873356ee6d1185ae2c8d'

/Users/cho/Developer/PMTMProject/LyricGeneration/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
import kagglehub
import pandas as pd
import glob
import re
import csv

# 1. Kaggle 데이터셋 다운로드
print("--- 1. Kaggle 데이터셋 로드 중 ---")
path1 = kagglehub.dataset_download("maharshipandya/-spotify-tracks-dataset")
path2 = kagglehub.dataset_download("rodolfofigueroa/spotify-12m-songs")

df1_raw = pd.read_csv(glob.glob(path1 + '/*.csv')[0])
df2_raw = pd.read_csv(glob.glob(path2 + '/*.csv')[0])

# 2. 피처 컬럼 추출 및 통합 사전 구성
ref1 = df1_raw[['track_name', 'artists', 'tempo', 'energy', 'danceability', 'loudness', 'valence']].copy()
ref1.columns = ['title', 'artist', 'bpm', 'energy', 'danceability', 'loudness', 'valence']

ref2 = df2_raw[['name', 'artists', 'tempo', 'energy', 'danceability', 'loudness', 'valence']].copy()
ref2.columns = ['title', 'artist', 'bpm', 'energy', 'danceability', 'loudness', 'valence']

big_ref = pd.concat([ref1, ref2]).drop_duplicates(subset=['title', 'artist'])

# 3. 매칭용 정제 함수
def super_clean(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'\(.*?\)|\[.*?\]', '', text)
    text = re.sub(r'[^a-zA-Z0-9가-힣\s]', '', text)
    return " ".join(text.split()).lower()

big_ref['match_key'] = big_ref['title'].apply(super_clean)

# 4. 가사 파일 로드
print("--- 2. 피처 매칭 시작 ---")
my_lyrics = pd.read_csv("korean_rap_lyrics.csv")  # 지니어스에서 수집한 가사 파일
my_lyrics['match_key'] = my_lyrics['title'].apply(super_clean)
print(f"대상: {len(my_lyrics)}곡")

# 5. 제목 기준 병합
df_master = pd.merge(
    my_lyrics,
    big_ref.drop(columns=['title', 'artist']),
    on='match_key',
    how='left'
)
df_master = df_master.drop_duplicates(subset=['spotify_id'])
df_master.drop(columns=['match_key'], inplace=True)

# 6. 결과 확인
matched_count = df_master['bpm'].notna().sum()
print(f"완료: {len(df_master)}곡 중 {matched_count}곡의 오디오 피처 확보!")
print(f"포함된 컬럼: {df_master.columns.tolist()}")

--- 1. Kaggle 데이터셋 로드 중 ---


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8.17M/8.17M [00:00<00:00, 12.8MB/s]

Extracting files...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 97.4M/97.4M [00:04<00:00, 24.8MB/s]

Extracting files...


--- 2. 피처 매칭 시작 ---
대상: 347곡
완료: 347곡 중 187곡의 오디오 피처 확보!
포함된 컬럼: ['title', 'artist', 'spotify_id', 'lyrics', 'bpm', 'energy', 'danceability', 'loudness', 'valence']


In [12]:
print(f"전체 곡 수: {total_rows}곡")
print(f"가사 확보: {lyrics_matched}곡 ({(lyrics_matched/total_rows)*100:.1f}%)")
print(f"오디오 피처(Kaggle) 매칭: {bpm_matched}곡 ({(bpm_matched/total_rows)*100:.1f}%)")

print("\n" + "="*30)

# 3. 어떤 아티스트의 데이터가 누락되었는지 확인 (중요)
# Kaggle 데이터와 매칭되지 않은 곡들을 아티스트별로 집계
missing_features_by_artist = df_master[df_master['bpm'].isna()].groupby('artist')['title'].count()
if not missing_features_by_artist.empty:
    print("⚠️ 오디오 피처가 누락된 아티스트별 곡 수:")
    print(missing_features_by_artist.sort_values(ascending=False))
else:
    print("✅ 모든 곡의 오디오 피처가 완벽하게 매칭되었습니다!")

NameError: name 'total_rows' is not defined

In [13]:
import csv

# 1. 결측치 포함 버전 (전체 데이터셋)
df_master.to_csv(
    "FINAL_DATASET_FULL.csv", 
    index=False, 
    encoding='utf-8-sig', 
    quoting=csv.QUOTE_ALL
)

# 2. 결측치 제거 버전
# BPM, Energy 등 오디오 피처가 NaN인 행을 모두 삭제합니다.
df_cleaned = df_master.dropna(subset=['bpm', 'energy', 'danceability', 'loudness', 'valence'])

df_cleaned.to_csv(
    "FINAL_DATASET_CLEANED.csv", 
    index=False, 
    encoding='utf-8-sig', 
    quoting=csv.QUOTE_ALL
)

# 결과 출력
print("--- 데이터셋 저장 완료 ---")
print(f"1. 전체 버전 (FULL): {len(df_master)}곡 저장 완료")
print(f"2. 정제 버전 (CLEANED): {len(df_cleaned)}곡 저장 완료")
print(f"   -> 피처 매칭 실패로 {len(df_master) - len(df_cleaned)}곡이 제외되었습니다.")

--- 데이터셋 저장 완료 ---
1. 전체 버전 (FULL): 347곡 저장 완료
2. 정제 버전 (CLEANED): 187곡 저장 완료
   -> 피처 매칭 실패로 160곡이 제외되었습니다.


In [14]:
# 1. 오디오 피처(Kaggle 데이터)가 없는 150곡만 따로 추출
# bpm이 NaN(결측치)인 데이터만 필터링합니다.
df_missing_only = df_master[df_master['bpm'].isna()].copy()

df_missing_only.to_csv(
    "MISSING_FEATURES_LIST.csv", 
    index=False, 
    encoding='utf-8-sig', 
    quoting=csv.QUOTE_ALL
)
print(f"--- 결측치 전용 데이터셋 생성 완료 ---")
print(f"파일명: MISSING_FEATURES_LIST.csv")
print(f"추출된 곡 수: {len(df_missing_only)}곡")
print(f"특이사항: 가사 수집은 완료되었으나 Kaggle 피처(BPM 등)가 없는 목록입니다.")

print("\n[아티스트별 누락 곡 수]")
print(df_missing_only.groupby('artist')['title'].count().sort_values(ascending=False))

--- 결측치 전용 데이터셋 생성 완료 ---
파일명: MISSING_FEATURES_LIST.csv
추출된 곡 수: 160곡
특이사항: 가사 수집은 완료되었으나 Kaggle 피처(BPM 등)가 없는 목록입니다.

[아티스트별 누락 곡 수]
artist
The Quiett         22
Simon Dominic      20
SUPERBEE           18
DOK2               16
Sik-K              16
Swings             13
UNEDUCATED KID     12
pH-1               12
Coogie             10
Jay Park            9
Paloalto            5
Loco                4
ASH ISLAND          2
죠지, 쿠기 (Coogie)     1
Name: title, dtype: int64
